# burned - control

In [18]:
import glob
import numpy as np
from tqdm import tqdm
from scipy.stats import mannwhitneyu, pearsonr
import os, sys
from typing import Dict, Tuple, List, Optional

project_dir = f'/home/{os.environ["USER"]}/FireScope'
os.chdir(project_dir)
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

from config import DATA_DIR

# ----------------------- I/O helpers -----------------------

def _load_npy_2d(path: str) -> np.ndarray:
    """Load .npy and return a 2D float32 array clipped to [0,1]."""
    arr = np.load(path)
    if arr.ndim == 3:
        arr = arr[0]
    elif arr.ndim != 2:
        raise ValueError(f"Unsupported shape {arr.shape} in {path}")
    if not np.isfinite(arr).all():
        arr = np.nan_to_num(arr, nan=0.0, posinf=1.0, neginf=0.0)
    # Map possible [-1,1] to [0,1], else clip.
    arr = np.clip(arr, 0.0, 1.0)
    return arr.astype(np.float32)

def _load_mask_2d(path: str) -> np.ndarray:
    """Load mask and return 2D uint8 binary array {0,1}."""
    m = np.load(path)
    if m.ndim == 3:
        m = m[0]
    elif m.ndim != 2:
        raise ValueError(f"Unsupported mask shape {m.shape} in {path}")
    m = (m > 0.5).astype(np.uint8)
    return m

def _index_by_basename(root: str):
    """Recursively collect all .npy files, indexed by basename. Raises if duplicates."""
    files = glob.glob(os.path.join(root, "**", "*.npy"), recursive=True)
    idx = {}
    for f in files:
        name = os.path.basename(f)
        if name in idx:
            raise ValueError(f"Duplicate filename detected for basename '{name}':\n - {idx[name]}\n - {f}")
        idx[name] = f
    return idx

# ----------------------- Core computation -----------------------

def compute_burn_control_stats(masks_dir: str,
                               burned_preds_dir: str,
                               control_preds_dir: str):
    # Index files
    mask_idx   = _index_by_basename(masks_dir)
    burned_idx = _index_by_basename(burned_preds_dir)
    control_files = glob.glob(os.path.join(control_preds_dir, "**", "*.npy"), recursive=True)

    # Sanity: masks and burned predictions must match 1:1 by basename
    if set(mask_idx.keys()) != set(burned_idx.keys()):
        missing_in_burned = sorted(set(mask_idx) - set(burned_idx))
        missing_in_masks  = sorted(set(burned_idx) - set(mask_idx))
        raise ValueError(
            "Masks and burned predictions file sets do not match.\n"
            f"Missing in burned: {missing_in_burned[:5]}{' ...' if len(missing_in_burned) > 5 else ''}\n"
            f"Missing in masks : {missing_in_masks[:5]}{' ...' if len(missing_in_masks) > 5 else ''}"
        )

    # Accumulators
    burned_sum_all, burned_count_all = 0.0, 0
    control_sum_all, control_count_all = 0.0, 0
    masked_sum, masked_count = 0.0, 0
    unmasked_sum, unmasked_count = 0.0, 0
    burned_means_per_image = []
    masked_means_per_image = []
    unmasked_means_per_image = []
    control_means_per_image = []

    # For metrics: predictions and labels (ignoring masks)
    burned_preds_per_image = []   # y=1
    control_preds_per_image = []  # y=0

    # 1) Burned predictions: global mean + masked/unmasked means (with masks)
    names = sorted(burned_idx.keys())
    for name in tqdm(names, desc="Burned preds + masks"):
        pred = _load_npy_2d(burned_idx[name])
        mask = _load_mask_2d(mask_idx[name])
        if pred.shape != mask.shape:
            raise ValueError(f"Shape mismatch for {name}: pred {pred.shape} vs mask {mask.shape}")

        # global (pixelwise)
        burned_sum_all += float(pred.sum())
        burned_count_all += int(pred.size)

        # masked/unmasked (pixelwise)
        m = (mask == 1)
        um = ~m
        if m.any():
            masked_sum += float(pred[m].sum())
            masked_count += int(m.sum())
        if um.any():
            unmasked_sum += float(pred[um].sum())
            unmasked_count += int(um.sum())

        # Per-image means
        img_mean = float(pred.mean())
        burned_means_per_image.append(img_mean)
        burned_preds_per_image.append(img_mean)  # for metrics
        masked_means_per_image.append(pred[m].mean() if m.any() else np.nan)
        unmasked_means_per_image.append(pred[um].mean() if um.any() else np.nan)

    # 2) Random control predictions: global mean + per-image means
    for f in tqdm(control_files, desc="Control preds"):
        pred = _load_npy_2d(f)
        control_sum_all += float(pred.sum())
        control_count_all += int(pred.size)
        img_mean = float(pred.mean())
        control_means_per_image.append(img_mean)
        control_preds_per_image.append(img_mean)  # for metrics

    # Safeguards
    if burned_count_all == 0 or control_count_all == 0:
        raise ValueError("No pixels found in one of the sets.")
    if masked_count == 0:
        raise ValueError("Masked pixel count is zero.")
    if unmasked_count == 0:
        raise ValueError("Unmasked pixel count is zero.")

    # Means (pixelwise)
    burned_mean_all = burned_sum_all / burned_count_all
    control_mean_all = control_sum_all / control_count_all
    diff_all = burned_mean_all - control_mean_all
    masked_mean = masked_sum / masked_count
    unmasked_mean = unmasked_sum / unmasked_count
    diff_masks = masked_mean - unmasked_mean

    # Print basic averages (unchanged)
    print("\n=== Averages ===")
    print(f"Burned predictions (all pixels): {burned_mean_all:.6f}")
    print(f"Random controls  (all pixels):   {control_mean_all:.6f}")
    print(f"Difference (burned - control):   {diff_all:.6f}")
    print(f"Burned predictions (masked):     {masked_mean:.6f}")
    print(f"Burned predictions (unmasked):   {unmasked_mean:.6f}")
    print(f"Difference (masked - unmasked):   {diff_masks:.6f}")

    # Build predictions/labels (per-image level)
    preds = np.array(burned_preds_per_image + control_preds_per_image, dtype=float)
    labels = np.concatenate([
        np.ones(len(burned_preds_per_image), dtype=int),
        np.zeros(len(control_preds_per_image), dtype=int)
    ])

    return {
        "burned_mean_all": burned_mean_all,
        "control_mean_all": control_mean_all,
        "diff_burned_minus_control": diff_all,
        "burned_masked_mean": masked_mean,
        "burned_unmasked_mean": unmasked_mean,
        "burned_means_per_image": burned_means_per_image,
        "masked_means_per_image": masked_means_per_image,
        "unmasked_means_per_image": unmasked_means_per_image,
        "control_means_per_image": control_means_per_image,
        # New keys for metrics:
        "preds": preds,     # per-image mean, in [0,1]
        "labels": labels,   # 1 for burned, 0 for control
    }

# ----------------------- Metrics & summary table -----------------------

def _roc_auc_from_scores(y: np.ndarray, p: np.ndarray) -> float:
    """ROC AUC via Mann–Whitney: P(p_pos > p_neg)."""
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    pos = p[y == 1]
    neg = p[y == 0]
    if len(pos) == 0 or len(neg) == 0:
        return np.nan
    U = mannwhitneyu(pos, neg, alternative="greater", method="asymptotic").statistic
    return float(U / (len(pos) * len(neg)))

def _brier(y: np.ndarray, p: np.ndarray) -> float:
    y = np.asarray(y, dtype=float); p = np.asarray(p, dtype=float)
    return float(np.mean((p - y) ** 2))

def _logloss(y: np.ndarray, p: np.ndarray, eps: float = 1e-12) -> float:
    y = np.asarray(y, dtype=float); p = np.asarray(p, dtype=float)
    p = np.clip(p, eps, 1.0 - eps)
    return float(-np.mean(y * np.log(p) + (1.0 - y) * np.log(1.0 - p)))

def _pearson_corr(y: np.ndarray, p: np.ndarray) -> float:
    # Point-biserial correlation equals Pearson between binary y and continuous p
    if len(y) < 2 or np.all(y == y[0]):
        return np.nan
    r, _ = pearsonr(p, y)
    return float(r)

def _ece_mce(y: np.ndarray, p: np.ndarray, n_bins: int = 15) -> tuple[float, float]:
    """
    Expected Calibration Error (ECE) and Max Calibration Error (MCE).
    Bins are linear in [0,1]. Returns (ECE, MCE).
    """
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    # Guard: if predictions are not in [0,1], clip (your loader already clamps)
    p = np.clip(p, 0.0, 1.0)
    N = len(p)
    if N == 0:
        return np.nan, np.nan

    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    mce = 0.0
    for b in range(n_bins):
        lo, hi = edges[b], edges[b+1]
        mask = (p >= lo) & (p < hi) if b < n_bins - 1 else (p >= lo) & (p <= hi)
        nb = int(mask.sum())
        if nb == 0:
            continue
        conf = float(p[mask].mean())
        acc = float(y[mask].mean())  # observed frequency
        gap = abs(acc - conf)
        ece += (nb / N) * gap
        mce = max(mce, gap)
    return float(ece), float(mce)

def _flatten_valid(pred: np.ndarray, gt: np.ndarray, mask: Optional[np.ndarray] = None) -> Tuple[np.ndarray, np.ndarray]:
    """Return 1D arrays of pred probs and gt labels (0/1) with an optional valid mask."""
    p = pred.ravel()
    g = gt.ravel()
    if mask is not None:
        m = mask.astype(bool).ravel()
    else:
        # valid where gt is finite (and where pred is finite)
        m = np.isfinite(g) & np.isfinite(p)
    # ensure binary labels
    g = (g > 0.5).astype(np.uint8) if g.dtype.kind != 'b' else g.astype(np.uint8)
    return p[m], g[m]

def _f1_from_counts(tp, fp, fn):
    denom = (2*tp + fp + fn)
    return 0.0 if denom == 0 else (2*tp) / denom

def _metrics_at_threshold(scores: np.ndarray, labels: np.ndarray, thr: float):
    """Compute confusion + metrics at score >= thr => positive."""
    y_pred = (scores >= thr)
    y_true = labels.astype(bool)
    tp = int(np.sum(y_pred & y_true))
    fp = int(np.sum(y_pred & ~y_true))
    fn = int(np.sum(~y_pred & y_true))
    tn = int(np.sum(~y_pred & ~y_true))
    acc = (tp + tn) / max(1, len(labels))
    f1 = _f1_from_counts(tp, fp, fn)
    return tp, fp, fn, tn, acc, f1

def _binary_confusion(y_true: np.ndarray, y_prob: np.ndarray, thr: float) -> Dict[str, float]:
    y_pred = (y_prob >= thr).astype(np.uint8)
    tp = int(np.sum((y_pred == 1) & (y_true == 1)))
    tn = int(np.sum((y_pred == 0) & (y_true == 0)))
    fp = int(np.sum((y_pred == 1) & (y_true == 0)))
    fn = int(np.sum((y_pred == 0) & (y_true == 1)))
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-12)
    iou = tp / max(tp + fp + fn, 1)
    acc = (tp + tn) / max(tp + tn + fp + fn, 1)
    tnr = tn / max(tn + fp, 1)  # specificity
    return dict(tp=tp, tn=tn, fp=fp, fn=fn, precision=prec, recall=rec, f1=f1, iou=iou, accuracy=acc, specificity=tnr)

def _mcc_from_counts(tp: int, fp: int, fn: int, tn: int) -> float:
    """Matthews Correlation Coefficient from confusion counts."""
    num = (tp * tn) - (fp * fn)
    den = np.sqrt(max((tp + fp), 1) * max((tp + fn), 1) * max((tn + fp), 1) * max((tn + fn), 1))
    return 0.0 if den == 0 else float(num / den)

def _cohen_kappa_from_counts(tp: int, fp: int, fn: int, tn: int) -> float:
    """Cohen's kappa from confusion counts."""
    total = tp + fp + fn + tn
    if total == 0:
        return float('nan')
    po = (tp + tn) / total
    # expected agreement by chance from marginals
    p_yes_pred = (tp + fp) / total
    p_yes_true = (tp + fn) / total
    p_no_pred  = (tn + fn) / total
    p_no_true  = (tn + fp) / total
    pe = p_yes_pred * p_yes_true + p_no_pred * p_no_true
    denom = (1.0 - pe)
    return 0.0 if denom == 0 else float((po - pe) / denom)


def summarize_model_metrics(name_to_stats: dict[str, dict], n_bins: int = 15, sort_by='Correlation'):
    """
    Computes per-model metrics using the 'preds' and 'labels' returned by compute_burn_control_stats.

    Metrics (columns):
      - Correlation (Pearson r)          [higher is better]
      - Brier score                      [lower is better]
      - LogLoss                          [lower is better]
      - ROC AUC                          [higher is better]
      - ECE (Expected Calibration Error) [lower is better]
      - Reliability (MCE)                [lower is better]

    Prints a table with models as rows; best in each column is bold.
    """
    # Collect results
    models = list(name_to_stats.keys())
    metrics = {
        "Correlation": [],
        "Brier": [],
        "LogLoss": [],
        "ROC_AUC": [],
        "ECE": [],
        "Reliability(MCE)": [],
        "Delta": [],
        "Precision": [],
        "Recall": [],
        "f1": [],
        "IoU": [],
        "accuracy": [],
        "specificity": [],
        "mcc": [],
        "cohen_kappa": [],
    }

    for m in models:
        s = name_to_stats[m]
        y = np.asarray(s["labels"], dtype=int)
        p = np.asarray(s["preds"], dtype=float)
        r = _pearson_corr(y, p)
        br = _brier(y, p)
        ll = _logloss(y, p)
        auc = _roc_auc_from_scores(y, p)
        ece, mce = _ece_mce(y, p, n_bins=n_bins)


        # Confusion-based metrics @ threshold
        tp, fp, fn, tn, acc, f1 = _metrics_at_threshold(p, y, .5)
        cm = _binary_confusion(y, p, .5)
        mcc = _mcc_from_counts(tp, fp, fn, tn)
        kappa = _cohen_kappa_from_counts(tp, fp, fn, tn)

        metrics["Delta"].append(s["diff_burned_minus_control"])
        metrics["Correlation"].append(r)
        metrics["Brier"].append(br)
        metrics["LogLoss"].append(ll)
        metrics["ROC_AUC"].append(auc)
        metrics["ECE"].append(ece)
        metrics["Reliability(MCE)"].append(mce)
        metrics["Precision"].append(float(cm["precision"]))
        metrics["Recall"].append(float(cm["recall"]))
        metrics["f1"].append(float(cm["f1"]))
        metrics["IoU"].append(float(cm["iou"]))
        metrics["accuracy"].append(float(cm["accuracy"]))
        metrics["specificity"].append(float(cm["specificity"]))
        metrics["mcc"].append(float(mcc))
        metrics["cohen_kappa"].append(float(kappa))

    # Determine best per column (higher-is-better vs lower-is-better)
    higher_better = {"Correlation", "ROC_AUC", "Delta", "Precision", "Recall", "f1", "IoU",
                    "accuracy", "specificity", "mcc", "cohen_kappa"}
    lower_better  = {"Brier", "LogLoss", "ECE", "Reliability(MCE)", "recon", "ssim", "grad", "mse", "mae"}
    col_names = list(metrics.keys())

    # Compute best values
    best_vals = {}
    for col in col_names:
        vals = np.array(metrics[col], dtype=float)
        if col in higher_better:
            best = np.nanmax(vals)
        else:
            best = np.nanmin(vals)
        best_vals[col] = best

    # Pretty print with bold best; ANSI bold if TTY, else **markdown**
    use_ansi = hasattr(sys.stdout, "isatty") and sys.stdout.isatty()
    def bold(s: str) -> str:
        return f"\033[1m{s}\033[0m" if use_ansi else f"**{s}**"

    # Order based in Reliability
    order = np.argsort(metrics[sort_by])
    if sort_by in higher_better:
        order = order[::-1]

    # Column widths
    model_w = max(5, max(len(m) for m in models)) + 2
    col_w = 12

    # Header
    header = f"{'Model':<{model_w}}"
    for col in col_names:
        header += f"{col:>{col_w}}"
    print("\n=== Metrics (burned vs control, per-image mean as prediction) ===")
    print(header)

    # Rows
    for i, m in zip(order, np.array(models)[order]):
        row = f"{m:<{model_w}}"
        for col in col_names:
            val = metrics[col][i]
            # format
            if np.isnan(val):
                s = "nan"
            else:
                s = f"{val:.3f}"
            # bold if best (with a tiny tolerance for ties)
            tol = 1e-12
            is_best = (
                (col in higher_better and np.isfinite(val) and (best_vals[col] - val) <= tol) or
                (col in lower_better and np.isfinite(val) and (val - best_vals[col]) <= tol)
            )
            row += f"{bold(s) if is_best else s:>{col_w}}"
        print(row)

    # Also return the raw numbers if you want to save/export later
    out = {m: {col: float(metrics[col][i]) for col in col_names} for i, m in enumerate(models)}
    return out


In [ ]:
unet_baseline = compute_burn_control_stats(
    masks_dir=f"{DATA_DIR}/europe_dataset/fire_masks",
    burned_preds_dir=f"{DATA_DIR}/evaluations/small/unet_baseline/europe_fires",
    control_preds_dir=f"{DATA_DIR}/evaluations/small/unet_baseline/europe_negatives",
)
unet_climate = compute_burn_control_stats(
    masks_dir=f"{DATA_DIR}/europe_dataset/fire_masks",
    burned_preds_dir=f"{DATA_DIR}/evaluations/small/unet_climate/europe_fires",
    control_preds_dir=f"{DATA_DIR}/evaluations/small/unet_climate/europe_negatives",
)
unet_oracle_no_cot = compute_burn_control_stats(
    masks_dir=f"{DATA_DIR}/europe_dataset/fire_masks",
    burned_preds_dir=f"{DATA_DIR}/evaluations/small/unet_oracle_no_cot/europe_fires",
    control_preds_dir=f"{DATA_DIR}/evaluations/small/unet_oracle_no_cot/europe_negatives",
)
unet_oracle = compute_burn_control_stats(
    masks_dir=f"{DATA_DIR}/europe_dataset/fire_masks",
    burned_preds_dir=f"{DATA_DIR}/evaluations/small/unet_oracle/europe_fires",
    control_preds_dir=f"{DATA_DIR}/evaluations/small/unet_oracle/europe_negatives",
)


alphaearth_baseline = compute_burn_control_stats(
    masks_dir=f"{DATA_DIR}/europe_dataset/fire_masks",
    burned_preds_dir=f"{DATA_DIR}/evaluations/small/alphaearth_baseline/europe_fires_ada",
    control_preds_dir=f"{DATA_DIR}/evaluations/small/alphaearth_baseline/europe_negatives",
)
alphaearth_climate = compute_burn_control_stats(
    masks_dir=f"{DATA_DIR}/europe_dataset/fire_masks",
    burned_preds_dir=f"{DATA_DIR}/evaluations/small/alphaearth_climate/europe_fires_ada",
    control_preds_dir=f"{DATA_DIR}/evaluations/small/alphaearth_climate/europe_negatives",
)
alphaearth_oracle_no_cot = compute_burn_control_stats(
    masks_dir=f"{DATA_DIR}/europe_dataset/fire_masks",
    burned_preds_dir=f"{DATA_DIR}/evaluations/small/alphaearth_oracle_no_cot/europe_fires_ada",
    control_preds_dir=f"{DATA_DIR}/evaluations/small/alphaearth_oracle_no_cot/europe_negatives",
)
alphaearth_oracle = compute_burn_control_stats(
    masks_dir=f"{DATA_DIR}/europe_dataset/fire_masks",
    burned_preds_dir=f"{DATA_DIR}/evaluations/small/alphaearth_oracle/europe_fires_ada",
    control_preds_dir=f"{DATA_DIR}/evaluations/small/alphaearth_oracle/europe_negatives",
)

segformer_baseline = compute_burn_control_stats(
    masks_dir=f"{DATA_DIR}/europe_dataset/fire_masks",
    burned_preds_dir=f"{DATA_DIR}/evaluations/small/segformer_baseline/europe_fires",
    control_preds_dir=f"{DATA_DIR}/evaluations/small/segformer_baseline/europe_negatives",
)
segformer_climate = compute_burn_control_stats(
    masks_dir=f"{DATA_DIR}/europe_dataset/fire_masks",
    burned_preds_dir=f"{DATA_DIR}/evaluations/small/segformer_climate/europe_fires",
    control_preds_dir=f"{DATA_DIR}/evaluations/small/segformer_climate/europe_negatives",
)
segformer_oracle_no_cot = compute_burn_control_stats(
    masks_dir=f"{DATA_DIR}/europe_dataset/fire_masks",
    burned_preds_dir=f"{DATA_DIR}/evaluations/small/segformer_oracle_no_cot/europe_fires",
    control_preds_dir=f"{DATA_DIR}/evaluations/small/segformer_oracle_no_cot/europe_negatives",
)
segformer_oracle = compute_burn_control_stats(
    masks_dir=f"{DATA_DIR}/europe_dataset/fire_masks",
    burned_preds_dir=f"{DATA_DIR}/evaluations/small/segformer_oracle/europe_fires",
    control_preds_dir=f"{DATA_DIR}/evaluations/small/segformer_oracle/europe_negatives",
)


qwen = compute_burn_control_stats(
    masks_dir=f"{DATA_DIR}/europe_dataset/fire_masks",
    burned_preds_dir=f"{DATA_DIR}/evaluations/small/qwen_decoder/europe_fires",
    control_preds_dir=f"{DATA_DIR}/evaluations/small/qwen_decoder/europe_negatives",
)
unet_large_baseline = compute_burn_control_stats(
    masks_dir=f"{DATA_DIR}/europe_dataset/fire_masks",
    burned_preds_dir=f"{DATA_DIR}/evaluations/large/unet_baseline/europe_fires",
    control_preds_dir=f"{DATA_DIR}/evaluations/large/unet_baseline/europe_negatives",
)
unet_large_climate = compute_burn_control_stats(
    masks_dir=f"{DATA_DIR}/europe_dataset/fire_masks",
    burned_preds_dir=f"{DATA_DIR}/evaluations/large/unet_climate/europe_fires",
    control_preds_dir=f"{DATA_DIR}/evaluations/large/unet_climate/europe_negatives",
)

In [ ]:
# Collect every model you computed into one dict
models_dict = {
    'unet_baseline': unet_baseline,
    'unet_climate': unet_climate,
    'unet_oracle_no_cot': unet_oracle_no_cot,
    'unet_oracle': unet_oracle,

    'alphaearth_baseline': alphaearth_baseline,
    'alphaearth_climate': alphaearth_climate,
    'alphaearth_oracle_no_cot': alphaearth_oracle_no_cot,
    'alphaearth_oracle': alphaearth_oracle,

    'segformer_baseline': segformer_baseline,
    'segformer_climate': segformer_climate,
    'segformer_oracle_no_cot': segformer_oracle_no_cot,
    'segformer_oracle': segformer_oracle,

    'Qwen+decoder': qwen,
    'unet^_baseline': unet_large_baseline,
    'unet^_climate': unet_large_climate,
}

import pandas as pd
metrics_table = summarize_model_metrics(models_dict, n_bins=15)
display(pd.DataFrame(metrics_table).T)


# masked - unmasked

In [8]:
import os, re, sys, math, json, time, random, glob
import numpy as np
from typing import Dict, Tuple, List, Optional
from scipy.stats import mannwhitneyu, pearsonr

# optional, for a high-quality SSIM if available
try:
    from skimage.metrics import structural_similarity as skimage_ssim
except Exception:
    skimage_ssim = None  # fall back to lightweight SSIM if skimage isn't available

project_dir = f'/home/{os.environ["USER"]}/FireScope'
os.chdir(project_dir)
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

from config import DATA_DIR
from tqdm import tqdm

# -------------------------------
# Helpers: loading & listing
# -------------------------------

def _load_npy(path: str) -> np.ndarray:
    arr = np.load(path)
    # ensure 2D: squeeze channel if saved as (1,H,W) or (C,H,W)
    if arr.ndim == 3:
        arr = arr[0]
    elif arr.ndim > 3:
        raise ValueError(f"Unsupported array shape {arr.shape} in {path}")
    # clip/normalize defensively to [0,1]
    if not np.isfinite(arr).all():
        arr = np.nan_to_num(arr, nan=0.0, posinf=1.0, neginf=0.0)
    # If values look like [-1,1], map to [0,1]
    if arr.min() < 0.0 and arr.max() <= 1.0:
        arr = np.clip(arr * 0.5 + 0.5, 0.0, 1.0)
    else:
        arr = np.clip(arr, 0.0, 1.0)
    return arr.astype(np.float32)

def _list_npy1(dir_path: str) -> List[str]:
    n = len(dir_path)
    if dir_path[-1] != '/':
        n += 1
    return sorted([f[n:] for f in glob.glob(os.path.join(dir_path,'*.npy'))])

def _list_npy2(dir_path: str) -> List[str]:
    n = len(dir_path)
    if dir_path[-1] != '/':
        n += 1
    return sorted([f[n:] for f in glob.glob(os.path.join(dir_path,'*/*.npy'))])

# -------------------------------
# Regression / similarity helpers
# -------------------------------

def _huber_loss(a: np.ndarray, b: np.ndarray, beta: float = 1.0) -> float:
    """Smooth L1 (Huber) with reduction='mean' like PyTorch SmoothL1Loss(beta)."""
    d = np.abs(a - b)
    quad = np.minimum(d, beta)
    # 0.5 * (quad^2) / beta  +  (d - quad)
    loss = 0.5 * (quad ** 2) / beta + (d - quad)
    return float(np.mean(loss))

def _mse(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.mean((a - b) ** 2))

def _mae(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.mean(np.abs(a - b)))

def _finite_diff_grads(x: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    Simple forward differences. x expected 2D (H, W).
    Returns (gx, gy) with the same shape as x (pad last row/col with zeros).
    """
    gx = np.zeros_like(x)
    gy = np.zeros_like(x)
    gx[:, :-1] = x[:, 1:] - x[:, :-1]
    gy[:-1, :] = x[1:, :] - x[:-1, :]
    return gx, gy

def _grad_loss(a: np.ndarray, b: np.ndarray) -> float:
    """
    L1 difference between image gradients, averaged.
    Matches the intent of many grad losses used during training.
    """
    ax, ay = _finite_diff_grads(a)
    bx, by = _finite_diff_grads(b)
    return float(np.mean(np.abs(ax - bx)) + np.mean(np.abs(ay - by)))

def _ssim(a: np.ndarray, b: np.ndarray) -> float:
    """
    SSIM in [0,1]. If skimage is available, use it; else use a lightweight fallback.
    Inputs are expected in [0,1]. Returns the SSIM (not 1-SSIM).
    """
    a = a.astype(np.float32)
    b = b.astype(np.float32)

    if skimage_ssim is not None:
        try:
            # modern skimage uses channel_axis; our arrays are 2D
            return float(skimage_ssim(a, b, data_range=1.0))
        except Exception:
            pass

    # Fallback: simplified SSIM (no Gaussian weighting)
    C1 = (0.01 ** 2)
    C2 = (0.03 ** 2)

    mu_a = np.mean(a)
    mu_b = np.mean(b)
    sigma_a2 = np.var(a)
    sigma_b2 = np.var(b)
    sigma_ab = np.mean((a - mu_a) * (b - mu_b))

    num = (2 * mu_a * mu_b + C1) * (2 * sigma_ab + C2)
    den = (mu_a ** 2 + mu_b ** 2 + C1) * (sigma_a2 + sigma_b2 + C2)
    if den == 0:
        return 1.0 if num == 0 else 0.0
    return float(num / den)

# -------------------------------
# Classification helpers (fast; no AUC)
# -------------------------------

def _flatten_valid(pred: np.ndarray, gt: np.ndarray, mask: Optional[np.ndarray] = None) -> Tuple[np.ndarray, np.ndarray]:
    """Return 1D arrays of pred probs and gt labels (0/1) with an optional valid mask."""
    p = pred.ravel()
    g = gt.ravel()
    if mask is not None:
        m = mask.astype(bool).ravel()
    else:
        # valid where gt is finite (and where pred is finite)
        m = np.isfinite(g) & np.isfinite(p)
    # ensure binary labels
    g = (g > 0.5).astype(np.uint8) if g.dtype.kind != 'b' else g.astype(np.uint8)
    return p[m], g[m]

def _binary_confusion(y_true: np.ndarray, y_prob: np.ndarray, thr: float) -> Dict[str, float]:
    y_pred = (y_prob >= thr).astype(np.uint8)
    tp = int(np.sum((y_pred == 1) & (y_true == 1)))
    tn = int(np.sum((y_pred == 0) & (y_true == 0)))
    fp = int(np.sum((y_pred == 1) & (y_true == 0)))
    fn = int(np.sum((y_pred == 0) & (y_true == 1)))
    # prec = tp / max(tp + fp, 1)
    # rec = tp / max(tp + fn, 1)
    # f1 = 2 * prec * rec / max(prec + rec, 1e-12)
    iou = tp / max(tp + fp + fn, 1)
    # acc = (tp + tn) / max(tp + tn + fp + fn, 1)
    # tnr = tn / max(tn + fp, 1)  # specificity
    return dict(tp=tp, tn=tn, fp=fp, fn=fn,  iou=iou, )

def _roc_auc_from_scores(y: np.ndarray, p: np.ndarray) -> float:
    """ROC AUC via Mann–Whitney: P(p_pos > p_neg)."""
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    pos = p[y == 1]
    neg = p[y == 0]
    if len(pos) == 0 or len(neg) == 0:
        return np.nan
    U = mannwhitneyu(pos, neg, alternative="greater", method="asymptotic").statistic
    return float(U / (len(pos) * len(neg)))

def _brier(y: np.ndarray, p: np.ndarray) -> float:
    y = np.asarray(y, dtype=float); p = np.asarray(p, dtype=float)
    return float(np.mean((p - y) ** 2))


def _ece_mce(y: np.ndarray, p: np.ndarray, n_bins: int = 15) -> tuple[float, float]:
    """
    Expected Calibration Error (ECE) and Max Calibration Error (MCE).
    Bins are linear in [0,1]. Returns (ECE, MCE).
    """
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    # Guard: if predictions are not in [0,1], clip (your loader already clamps)
    p = np.clip(p, 0.0, 1.0)
    N = len(p)
    if N == 0:
        return np.nan, np.nan

    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    mce = 0.0
    for b in range(n_bins):
        lo, hi = edges[b], edges[b+1]
        mask = (p >= lo) & (p < hi) if b < n_bins - 1 else (p >= lo) & (p <= hi)
        nb = int(mask.sum())
        if nb == 0:
            continue
        conf = float(p[mask].mean())
        acc = float(y[mask].mean())  # observed frequency
        gap = abs(acc - conf)
        ece += (nb / N) * gap
        mce = max(mce, gap)
    return float(ece), float(mce)

def _append_classification_metrics(out: Dict[str, float],
                                   all_probs: np.ndarray,
                                   all_labels: np.ndarray,
                                   threshold: float = 0.5,
                                   verbose: bool = True, n_bins=15) -> Dict[str, float]:
    """
    Fast classification/probabilistic metrics (no ROC/PR curves, no AUC):
      - Brier score
      - Confusion-matrix metrics @ threshold: precision, recall, F1, IoU, accuracy, specificity
      - MCC, Cohen's kappa
    """
    # Brier on all valid pixels
    cm = _binary_confusion(all_labels, all_probs, threshold)

    y = all_labels
    p = all_probs
    br = _brier(y, p)
    auc = _roc_auc_from_scores(y, p)
    ece, mce = _ece_mce(y, p, n_bins=n_bins)

    out.update({
        "Brier": float(br),
        "ROC_AUC": float(auc),
        "ECE": float(ece),
        "iou@thr": float(cm["iou"]),
    })

    return out

# -------------------------------
# Public API
# -------------------------------

def get_metrics(pred_path, gt_path,
                add_curves: bool = True,
                threshold: float = 0.5,
                mask_dir: Optional[str] = None,
                add_control: bool = True
            ) -> Dict[str, float]:
    """
    Extended version (fast):
      - add_curves: if True, computes fast classification metrics (no ROC/PR curves, no AUC)
      - threshold:  cutoff for binarizing predictions (for confusion-matrix metrics)
      - mask_dir:   optional directory of boolean masks (same filenames) where True=valid pixels
    """
    pred_items = _list_npy2(pred_path)
    if add_control:
        if 'europe_fires_ada' in pred_path:
            pred_items += _list_npy1(pred_path.replace('europe_fires_ada','europe_negatives'))
        else:
            pred_items += _list_npy1(pred_path.replace('europe_fires','europe_negatives'))
    gt_items = _list_npy2(gt_path)
    if add_control:
        gt_items += _list_npy1(gt_path.replace('fire_masks','no_fire_masks'))

    pred_items = [p for p in pred_items]
    items = list(set.intersection(set(pred_items), set(gt_items)))
    # print(len(items))
    if len(items) == 0:
        raise ValueError("No .npy files found.")

    totals = {}
    n = 0
    all_probs_list, all_labels_list = [], []

    for name in items:
        if 'NN_negative' in name:
            if 'europe_fires_ada' in pred_path:
                pth = pred_path.replace('europe_fires_ada','europe_negatives')
            else:
                pth = pred_path.replace('europe_fires','europe_negatives')
            pred = _load_npy(os.path.join(pth, name))  # [0,1] prob map
            gt   = _load_npy(os.path.join(gt_path.replace('fire_masks','no_fire_masks'), name))    # assumed binary (0/1) or continuous then binarized (>0.5)
        else:
            pred = _load_npy(os.path.join(pred_path, name))  # [0,1] prob map
            gt   = _load_npy(os.path.join(gt_path, name))    # assumed binary (0/1) or continuous then binarized (>0.5)

        if pred.shape != gt.shape:
            raise ValueError(f"Shape mismatch for {name}: pred {pred.shape} vs gt {gt.shape}")

        # optional validity mask
        mask = None
        if mask_dir:
            mask_fp = os.path.join(mask_dir, name)
            if os.path.exists(mask_fp):
                mask = np.load(mask_fp).astype(bool)
                if mask.shape != pred.shape:
                    raise ValueError(f"Mask shape mismatch for {name}: {mask.shape} vs {pred.shape}")

        # --- regression-style losses on continuous maps ---
        n += 1

        # --- collect for fast classification metrics (prob/label vectors) ---
        if add_curves:
            probs_flat, labels_flat = _flatten_valid(pred, gt, mask)
            all_probs_list.append(probs_flat)
            all_labels_list.append(labels_flat)

    out = {k: (v / max(n, 1)) for k, v in totals.items()}

    # Add fast classification metrics (no AUC/curves)
    if add_curves and all_probs_list:
        all_probs = np.concatenate(all_probs_list)
        all_labels = np.concatenate(all_labels_list)
        out = _append_classification_metrics(out, all_probs, all_labels,
                                             threshold=threshold, verbose=False)

    return out


# --- Multi-folder evaluation + pretty table with bold best-per-metric ---

import os
import pandas as pd
from typing import List, Optional, Dict

# Assumes get_metrics(...) is already defined in the notebook (from your code)

def evaluate_prediction_folders(
    gt_dir: str,
    pred_dirs: Dict[str, str],
    threshold: float = 0.5,
    mask_dir: Optional[str] = None,
    add_classification_metrics: bool = True,
    add_control = False,
) -> pd.DataFrame:
    """
    Runs get_metrics for each prediction folder and returns a DataFrame with rows = model (folder name),
    columns = metrics. Folder names are basename-only (no full paths).
    """
    rows = []

    for name in tqdm(pred_dirs):
        pdir = pred_dirs[name]
        out = get_metrics(
            pred_path=pdir,
            gt_path=gt_dir,
            add_curves=add_classification_metrics,
            threshold=threshold,
            mask_dir=mask_dir,
            add_control=add_control
        )
        rows.append(out)

    df = pd.DataFrame(rows, index=pred_dirs.keys())

    # Put columns in a consistent, helpful order if present
    preferred_order = [
        "Brier","ROC_AUC","ECE","IoU",
    ]
    cols = [c for c in preferred_order if c in df.columns] + [c for c in df.columns if c not in preferred_order]
    df = df[cols]
    return df


def style_bold_best(df: pd.DataFrame, precision=4):
    """
    Bold the best value per column:
      - For 'higher is better' metrics: bold the max.
      - For 'lower is better' metrics: bold the min.
    """
    higher_is_better = {"Correlation", "ROC_AUC", "Delta", "Precision", "Recall", "f1", "IoU",
                    "accuracy", "specificity", "mcc", "cohen_kappa"}
    lower_is_better  = {"Brier", "LogLoss", "ECE", "Reliability(MCE)", "recon", "ssim", "grad", "mse", "mae"}

    def highlight(col: pd.Series) -> list:
        if col.name in higher_is_better:
            best = col.max(skipna=True)
            return ["font-weight: bold" if v == best else "" for v in col]
        elif col.name in lower_is_better:
            best = col.min(skipna=True)
            return ["font-weight: bold" if v == best else "" for v in col]
        else:
            return [""] * len(col)

    return (
        df.style
          .apply(highlight, axis=0)
          .format(precision=precision)
    )


In [30]:
# + conrols!!!
threshold = 0.5
mask_dir = None  # or "/path/to/masks" if you have valid masks
gt_dir = f"{DATA_DIR}/europe_dataset/fire_masks"
pred_dirs = {
    'unet_baseline':f"{DATA_DIR}/evaluations/small/unet_baseline/europe_fires",
    'unet_climate':f"{DATA_DIR}/evaluations/small/unet_climate/europe_fires",
    'unet_oracle':f"{DATA_DIR}/evaluations/small/oracle_unet_no_cot/europe_fires",
    'unet_cot_oracle':f"{DATA_DIR}/evaluations/small/oracle_unet/europe_fires",

    'alphaearth_baseline':f"{DATA_DIR}/evaluations/small/alphaearth_baseline/europe_fires_ada",
    'alphaearth_climate':f"{DATA_DIR}/evaluations/small/alphaearth_climate/europe_fires_ada",
    'alphaearth_oracle':f"{DATA_DIR}/evaluations/small/oracle_alphaearth_no_cot/europe_fires_ada",
    'alphaearth_cot_oracle':f"{DATA_DIR}/evaluations/small/oracle_alphaearth/europe_fires_ada",

    'segformer_baseline':f"{DATA_DIR}/evaluations/small/segformer_baseline/europe_fires",
    'segformer_climate':f"{DATA_DIR}/evaluations/small/segformer_climate/europe_fires",
    'segformer_oracle':f"{DATA_DIR}/evaluations/small/oracle_segformer_no_cot/europe_fires",
    'segformer_cot_oracle':f"{DATA_DIR}/evaluations/small/oracle_segformer/europe_fires",

    'qwen-decoder':f"{DATA_DIR}/evaluations/small/qwen_vlm_decoder_head/europe_fires",
    'unet^_baseline':f"{DATA_DIR}/evaluations/small/big/unet_baseline/europe_fires",
    'unet^_climate':f"{DATA_DIR}/evaluations/small/big/unet_climate/europe_fires",
}
df = evaluate_prediction_folders(gt_dir, pred_dirs, threshold=threshold, mask_dir=mask_dir, add_control=False)
styler = style_bold_best(df)
display(styler)